## Assignment Module 8 Lab 1: Predicting House Prices in California using Linear Regression

### Background

Accurately predicting house prices is essential in the real estate market. In this assignment, you will use linear regression to predict house prices based on various features such as the number of rooms, population density, and median income in different neighborhoods in California.

### Dataset

The dataset you will be using is the California Housing dataset, which contains information about houses in different locations in California. The dataset includes 20,640 observations and 8 variables, including:

- `MedInc`: Median income in the block
- `HouseAge`: Median house age in the block
- `AveRooms`: Average number of rooms per dwelling
- `AveBedrms`: Average number of bedrooms per dwelling
- `Population`: Block population
- `AveOccup`: Average house occupancy
- `Latitude`: Latitude coordinate of the block
- `Longitude`: Longitude coordinate of the block

You can load the dataset using the following code:

```python
from sklearn.datasets import fetch_california_housing

# Load the California Housing dataset
california = fetch_california_housing(as_frame=True)

# Access the data and target variables
X = california.data
y = california.target
```

### Task

Your task is to predict house prices in California using linear regression. You will need to perform the following steps:

1. Load the California Housing dataset into a Pandas DataFrame.
2. Split the dataset into training and testing sets, with a 70/30 split.
3. Fit a `LinearRegression` model to the training data.
4. Predict house prices for the testing set.
5. Evaluate the performance of the model using the following error checking metrics:
   - R-squared (R2)
   - Mean Squared Error (MSE)
   - Mean Absolute Error (MAE)
   - Akaike Information Criterion (AIC)
6. Analyze the performance of the model and provide recommendations for improvement.

You can use the `OLS` (ordinary least squares) method from the `statsmodels.regression.linear_model` module to calculate the Akaike Information Criterion (AIC).

### Submission Instructions

Please submit a Jupyter notebook containing your code, output, and analysis. Make sure to include the following sections in your notebook:

1. Introduction
2. Dataset Description
3. Data Preprocessing
4. Model Training
5. Model Evaluation
6. Performance Analysis and Recommendations
7. Conclusion

Your notebook should be well-documented and easy to follow, with clear explanations of the steps you took and the results you obtained. Make sure to comment your code and include markdown cells explaining your thought process and conclusions. 

# Predicting California House Prices Using Linear Regression

## 1. Introduction 

This notebook predicts median house values across California using demographic, housing, and geographic features from the California Housing dataset. An analysis of a baseline Ordinary Least Squares (OLS) linear regression model, evaluating its predictive accuracy and parsimony using $R^2$, Mean Squared Error (MSE), Mean Absolute Error (MAE), and the Akaike Information Criterion (AIC) to establish a benchmark for future modeling iterations.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

# Load the California Housing dataset directly from the scikit-learn library
california_housing = fetch_california_housing(as_frame=True)
df = california_housing.frame

##  2. Dataset Description

In [2]:
print(california_housing.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib:
https://lib.stat.cmu.edu/datasets/houses.zip

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block g

##  3. Data Preprocessing  

The DataFrame is inspected to verify its dimensions, confirm that all features are stored as numeric data types (`df.dtypes`), and check for missing values or duplicate records. 

In [3]:
df.head()
df.info()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


np.int64(0)

All 20,640 records across 9 columns are stored as numeric datatypes, with no missing values and zero duplicate rows. Consequently, no imputation or categorical encoding is required, and features are left unscaled so the baseline OLS coefficients remain directly interpretable in their original units.  

Next, the predictors ($X$) are separated from the target variable (`MedHouseVal`) and the dataset will be split into `70% training` and `30% testing` sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_train.shape, X_test.shape

((14448, 8), (6192, 8))

## 4. Model Training 

The training datasets `X_train` and `y_train` are fit into a Linear Regression model to estimate the baseline linear relationship between the eight housing features and the median value, `MedHouseVal`.

In [5]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ 0.45, 0.01,-0.12,...,-0. ,-0.42,-0.43]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['MedInc','HouseAge','AveRooms',...,'AveOccup','Latitude','Longitude']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-37.06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(8)


## 5. Model Evaluation

The fitted Linear Regression model is applied to the unseen `30%` testing set `X_test`, to generate predicted median house values, `y_pred`.

Model performance is evaluated across four standard regression metrics:

* **R-squared ($R^2$):** The coefficient of determination measures how well observed outcomes are replicated based on the proportion of total variation in the target variable `MedHouseVal`, which is predictable from the independent predictor features (`MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, and `Longitude`) on the unseen testing set (`X_test`, `y_test`).
* **Mean Squared Error (MSE):** The average squared difference measures the error between the observed actual values (`y_test`) and their corresponding predicted values (`y_pred`), giving more weight to larger errors.
* **Mean Absolute Error (MAE):** The mean of the sum of absolute error provides a linear measure of the error between paired observations,(`y_pred`) and actual (`y_test`) house values in the original target units ($100,000s).
* **Akaike Information Criterion (AIC):** Estimates prediction error, revealing whether the model is suitable for the given dataset.  

In [6]:
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

# test data metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# training data AIC 
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit()
aic = ols_model.aic

pd.Series(
    {"R-squared (R2)": r2, "MSE": mse, "MAE": mae, "AIC": aic},
    name="Evaluation Metrics",
).to_frame()

,Evaluation Metrics
R-squared (R2),0.595770
MSE,0.530568
MAE,0.527247
AIC,31664.708783


## 6. Performance Analysis and Recommendations

The baseline model's predictive accuracy and error are examined to produce actionable recommendations for optimizing performance. 
This evaluation proceeds by inspecting the fitted Ordinary Least Squares (OLS) summary statistics (`ols_model.summary()`) to assess feature coefficients, statistical significance ($p$-values), and diagnostic indicators, then generating insights alongside the test-set evaluation metrics ($R^2$, MSE, MAE, and AIC) to identify underlying limitations and modeling improvements.

In [7]:
ols_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            MedHouseVal   R-squared:                       0.609
Model:                            OLS   Adj. R-squared:                  0.609
Method:                 Least Squares   F-statistic:                     2815.
Date:                Wed, 25 Nov 2026   Prob (F-statistic):               0.00
Time:                        16:49:43   Log-Likelihood:                -15823.
No. Observations:               14448   AIC:                         3.166e+04
Df Residuals:                   14439   BIC:                         3.173e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -37.0562      0.779    -47.577      0.000     -38.583     -35.530
MedInc         0.4458      0.005     89.283      0.000       0.436       0.456
HouseAge       0.0097      0.001     18.240      0.000       0.009       0.011
AveRooms      -0.1221      0.007    -17.362      0.000      -0.136      -0.108
AveBedrms      0.7786      0.035     21.978      0.000       0.709       0.848
Population -7.757e-07   5.63e-06     -0.138      0.890   -1.18e-05    1.03e-05
AveOccup      -0.0034      0.000     -6.889      0.000      -0.004      -0.002
Latitude      -0.4185      0.008    -49.265      0.000      -0.435      -0.402
Longitude     -0.4337      0.009    -48.757      0.000      -0.451      -0.416
==============================================================================
Omnibus:                     2973.049   Durbin-Watson:                   1.964
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             8566.499
Skew:                           1.081   Prob(JB):                         0.00
Kurtosis:                       6.091   Cond. No.                     2.37e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.37e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Performance Analysis

The OLS model accounts for 60.9% of the variance in `MedHouseVal` on the training set and an unseen test $R^2$ of 59.58% reveals that the model generalizes well without signs of overfitting. 
Leaving 40.42% of test price variance unexplained, a test MAE of $0.527247 \approx$ `$52, 725`, and an MSE of $0.530568 \approx$ `$53, 057`shows substantial prediction errors on high-value properties.

The OLS diagnostics clarifies what drives both the model's predictive power and its blind spots, where each coefficient ($\beta$) measures the expected change in `MedHouseVal` per one-unit increase in a predictor, holding all other features constant:

* **Economic and Spatial Drivers:** Median income (`MedInc`, $\beta = 0.4458, p < 0.001$) is the strongest positive determinant, adding roughly $44,580 in predicted home value per $10,000 in income. `Latitude` ($\beta = -0.4185$) and `Longitude` ($\beta = -0.4337$) show California's southwest-to-northeast coastal price gradient as both negative coefficients, meaning predicted home prices rise as latitude decreases (moving toward southern california), and as longitude becomes more negative (moving toward the pacific coast). However, this highlights a model limitation as it cannot isolate localized high-priced coastal areas like San Francisco (north), and lower-priced inland calley cities, at the same latitiude. `Population` ($p = 0.890$) contributes no statistically meaningful linear signal.
  
* **Multicollinearity and Scale Distortion:** A large condition number of $2.37 \times 10^5$, implies coefficient estimates are highly sensitive to small input fluctuations due to unscaled features and overlapping predictors. Wide scale disparities such as high `Population` and single-digit room ratios in combination with strong collinearity between `AveRooms` ($\beta = -0.1221$) and `AveBedrms` ($\beta = +0.7786$), where total room count has to take counterintuitive negative slope to keep the number of bedrooms constant. 

* **Non-Normal Residuals and Target Capping:** Positive skew ($1.081$), heavy tails (Kurtosis = $6.091$), and a Jarque-Bera statistic of $8566.50$ ($p < 0.001$) demonstrates that errors do not follow a normal distribution, largely because of a $500,001 upper-bound truncation (`MedHouseVal = 5.00001`), where linear projections systematically underpredict luxury coastal markets.

### Recommendations for Improvement

1. **Engineer Ratios and Spatial Features:** Construct new predictors like giving AveRooms and AveBedrms their own ratio (`AveBedrms / AveRooms`), remove insignificant predictors like `Population`, and add distance-to-coast or distance-to-metro-hub features for spatial effects aren't captured from coordinates alone.
2. **Standardize Scales and Apply Regularization:** Fit `StandardScaler` with only `X_train` within a `scikit-learn` pipeline to rid of scale-driven numerical instability, and to stabilize collinear feature weights, test with **Ridge** or **ElasticNet** regression.
3. **Adopt Non-Linear Ensemble Models:** Tree-based ensembles like **Random Forest**, **Gradient Boosting**, or **XGBoost** that are  capable of capturing non-linear interactions between income, location, and housing density without assuming a strictly linear relationship.
4. **Address Right-Censoring and Target Skew:** Evaluate with a log-transformed target $\log(MedHouseVal)$, or run a sensitivity check without the $500,001, preventing upper-bound ceiling distortion from biasing slope estimates.

## 7. Conclusion

An Ordinary Least Squares (OLS) baseline model was trained and tested on the California Housing dataset as an interpretable regional benchmark for predicting median home values. With 6192 unseen test cases, the model reached an $R^2$ of `59.58%`, an MAE of $0.5272 \approx$ `$52,725`, an MSE of $0.530568 \approx$ `$53, 057`, and an in-sample AIC of nearly $31,664.71$.  

Although the model successfully indicates major economic and geographic trends, specifically a strong influence of household income and coastal price gradient, diagnostic checks show that real estate pricing here breaks several core linear assumptions. Overlapping housing ratios, unscaled features, and heavy-tailed errors caused by the $500,001 price cap limits the accuracy of a simple straight-line model.   

Overall, OLS works well as a transparent starting point for spotting key price drivers, while more accurate house valuation requires engineered features, standardized preprocessing pipelines, and non-linear models.